<a href="https://colab.research.google.com/github/thomaslu678/Praxis-Lab-25-26/blob/main/clean/15_Automate_GIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs

In [42]:
# ============================================================
# INSTALLS (Colab)
# ============================================================
!pip install geopandas shapely pyogrio pandas numpy pyproj centerline==1.1.1

# Imports

In [43]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import requests

import geopandas as gpd

from scipy.optimize import brentq

from shapely.geometry import (
    box,
    Point,
    MultiLineString
)

from centerline.geometry import Centerline

# Fetch all files

In [10]:
owner = "thomaslu678"
repo = "Praxis-Lab-25-26"
branch = "main"
parent_directory = "assets/QGIS Files"

url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"

In [11]:
response = requests.get(url)
response.raise_for_status()
tree = response.json()

# Store the folders immediately underneath parent_directory
subfolders = set()

for item in tree.get("tree", []):

    if item["type"] != "tree":
        continue

    path = item["path"]

    # Check whether this is an immediate child folder
    # of parent_directory
    if path.startswith(parent_directory + "/"):

        relative_path = path[len(parent_directory) + 1:]

        # Only keep the first directory level
        if "/" not in relative_path:
            subfolders.add(path)


print(f"Found {len(subfolders)} subfolders:\n")

for folder in sorted(subfolders):
    print(folder)

Found 23 subfolders:

assets/QGIS Files/0
assets/QGIS Files/1
assets/QGIS Files/10
assets/QGIS Files/11
assets/QGIS Files/12
assets/QGIS Files/13
assets/QGIS Files/14
assets/QGIS Files/15
assets/QGIS Files/16
assets/QGIS Files/17
assets/QGIS Files/18
assets/QGIS Files/19
assets/QGIS Files/2
assets/QGIS Files/20
assets/QGIS Files/21
assets/QGIS Files/22
assets/QGIS Files/3
assets/QGIS Files/4
assets/QGIS Files/5
assets/QGIS Files/6
assets/QGIS Files/7
assets/QGIS Files/8
assets/QGIS Files/9


In [12]:
# ---------------------------------------------------------
# Set up local download directory
# ---------------------------------------------------------

download_directory = "downloaded_gpkg"
os.makedirs(download_directory, exist_ok=True)

# Dictionary to store all GeoDataFrames
gdfs = {}

# ---------------------------------------------------------
# Loop through each subfolder
# ---------------------------------------------------------

for folder in sorted(subfolders):

    print(f"\n{'=' * 60}")
    print(f"Processing folder: {folder}")
    print(f"{'=' * 60}")

    # Find all files contained within this subfolder
    for item in tree.get("tree", []):

        if item["type"] != "blob":
            continue

        file_path = item["path"]

        # Is this file inside the current subfolder?
        if file_path.startswith(folder + "/"):

            print(f"  File found: {file_path}")

            # Only process GeoPackage files
            if file_path.lower().endswith(".gpkg"):

                print(f"  Downloading: {file_path}")

                # -------------------------------------------------
                # Download the GeoPackage from GitHub
                # -------------------------------------------------

                raw_url = (
                    f"https://raw.githubusercontent.com/"
                    f"{owner}/{repo}/{branch}/{file_path}"
                )

                file_response = requests.get(raw_url)
                file_response.raise_for_status()

                # -------------------------------------------------
                # Save the GeoPackage locally
                # -------------------------------------------------

                filename = os.path.basename(file_path)
                folder_name = os.path.basename(folder)

                local_folder = os.path.join(
                    download_directory,
                    folder_name
                )

                os.makedirs(local_folder, exist_ok=True)

                local_path = os.path.join(
                    local_folder,
                    filename
                )

                with open(local_path, "wb") as f:
                    f.write(file_response.content)

                print(f"  Saved to: {local_path}")

                # # -------------------------------------------------
                # # Read the GeoPackage with GeoPandas
                # # -------------------------------------------------

                # try:

                #     gdf = gpd.read_file(local_path)

                #     # Create a useful name for the GeoDataFrame
                #     # Example:
                #     # folder_01/data.gpkg
                #     # becomes:
                #     # gdfs["folder_01_data"]
                #     dataset_name = (
                #         f"{folder_name}_"
                #         f"{os.path.splitext(filename)[0]}"
                #     )

                #     gdfs[dataset_name] = gdf

                #     print(
                #         f"  Loaded successfully: "
                #         f"{dataset_name}"
                #     )

                #     print(
                #         f"  Rows: {len(gdf):,} | "
                #         f"Columns: {len(gdf.columns)}"
                #     )

                # except Exception as e:

                #     print(
                #         f"  ERROR reading {file_path}: {e}"
                #     )


Processing folder: assets/QGIS Files/0
  File found: assets/QGIS Files/0/Medium River.gpkg
  Downloading: assets/QGIS Files/0/Medium River.gpkg
  Saved to: downloaded_gpkg/0/Medium River.gpkg
  File found: assets/QGIS Files/0/Park.gpkg
  Downloading: assets/QGIS Files/0/Park.gpkg
  Saved to: downloaded_gpkg/0/Park.gpkg
  File found: assets/QGIS Files/0/Shape.gpkg
  Downloading: assets/QGIS Files/0/Shape.gpkg
  Saved to: downloaded_gpkg/0/Shape.gpkg

Processing folder: assets/QGIS Files/1
  File found: assets/QGIS Files/1/Shape.gpkg
  Downloading: assets/QGIS Files/1/Shape.gpkg
  Saved to: downloaded_gpkg/1/Shape.gpkg

Processing folder: assets/QGIS Files/10
  File found: assets/QGIS Files/10/Park.gpkg
  Downloading: assets/QGIS Files/10/Park.gpkg
  Saved to: downloaded_gpkg/10/Park.gpkg
  File found: assets/QGIS Files/10/Shape.gpkg
  Downloading: assets/QGIS Files/10/Shape.gpkg
  Saved to: downloaded_gpkg/10/Shape.gpkg

Processing folder: assets/QGIS Files/11
  File found: assets/QGIS

# Execute workflow

## Paths and helpers

In [851]:
# ============================================================
# USER INPUTS
# ============================================================

num = 9

case_study_path = "/content/downloaded_gpkg/" + str(num) + "/Shape.gpkg"

park_paths = []
park_paths = ["/content/downloaded_gpkg/" + str(num) + "/Park.gpkg",
              "/content/downloaded_gpkg/" + str(num) + "/Park 2.gpkg"]

water_paths = []
water_paths = ["/content/downloaded_gpkg/" + str(num) + "/Large River.gpkg"]

In [852]:
start_sub = "downloaded_gpkg/"
end_sub = "/"

# Find the positions of the delimiters
start_idx = case_study_path.find(start_sub)
end_idx = case_study_path.find(end_sub, start_idx + len(start_sub))

case_study_index = None

# Extract if both delimiters exist
if start_idx != -1 and end_idx != -1:
    case_study_index = case_study_path[start_idx + len(start_sub):end_idx]

In [853]:
output_dir = "/content/outputs/" + case_study_index + "/"

os.makedirs(output_dir, exist_ok=True)

output_gpkg = os.path.join(
    output_dir,
    "workflow_outputs.gpkg"
)

metadata_json = os.path.join(
    output_dir,
    "metadata.json"
)

# remove previous gpkg if rerunning

if os.path.exists(output_gpkg):
    os.remove(output_gpkg)

In [854]:
# ============================================================
# HELPERS
# ============================================================

def save_layer(gdf, layer_name):

    gdf.to_file(
        output_gpkg,
        layer=layer_name,
        driver="GPKG",
        index=False
    )

    print(f"Saved layer: {layer_name}")


def union_geometry(gdf):
    """
    Version-safe dissolve.
    """

    try:
        return gdf.union_all()
    except Exception:
        return gdf.unary_union

# Park Area must be in square meters
def get_cooling_extent_area(park_area):
    return 644.8 * (park_area)**(0.454)

water_cooling_extent_data = {
    ("Small", "River"): 668.7,
    ("Medium", "River"): 528.9,
    ("Large", "River"): 735.2,
    ("Small", "Lake"): 577.5,
    ("Medium", "Lake"): 559.7,
    ("Large", "Lake"): 904.4,
}

def get_water_cooling_extent(water_size, water_type):
    return water_cooling_extent_data[(water_size, water_type)]

## Inputs and buffers

In [855]:
# ============================================================
# LOAD INPUTS
# ============================================================

case_gdf = gpd.read_file(case_study_path)

parks_gdfs = []
for path in park_paths:
    parks_gdf = (
        gpd.read_file(path)
        if path
        else None
    )
    parks_gdfs.append(parks_gdf)


water_gdfs = []
for path in water_paths:
    water_gdf = (
        gpd.read_file(path)
        if path
        else None
    )

    start_idx = path.rfind('/') + 1
    end_idx = path.find(".gpkg")

    metadata = path[start_idx : end_idx].split()
    water_gdfs.append([water_gdf] + metadata)

crs = case_gdf.crs

print("CRS:", crs)

CRS: EPSG:32618


### Case

In [856]:
# ============================================================
# STEP 1
# AREA + PERIMETER
# ============================================================

case_gdf = case_gdf.copy()

case_gdf["fid"] = range(len(case_gdf))

case_gdf["area"] = case_gdf.geometry.area
case_gdf["perimeter"] = case_gdf.geometry.length

save_layer(
    case_gdf,
    "00_case_metrics"
)

total_area = float(
    case_gdf["area"].sum()
)

total_perimeter = float(
    case_gdf["perimeter"].sum()
)

print("\nCASE STUDY METRICS")
print("------------------")
print("Area:", total_area)
print("Perimeter:", total_perimeter)

Saved layer: 00_case_metrics

CASE STUDY METRICS
------------------
Area: 31841.34131599858
Perimeter: 4791.22757832132


In [857]:
# ============================================================
# STEP 2
# CENTERLINE LENGTH + SAVE CENTERLINE LAYER
# ============================================================

case_polygon = case_gdf.geometry.iloc[0]

# repair geometry if needed
case_polygon = case_polygon.buffer(0)

centerline_length = None

try:

    centerline_obj = Centerline(
        case_polygon
    )

    centerline_geom = centerline_obj.geometry


    # save centerline as GeoPackage layer
    centerline_gdf = gpd.GeoDataFrame(
        {
            "fid": [0],
            "length": [
                float(centerline_geom.length)
            ]
        },
        geometry=[centerline_geom],
        crs=crs
    )


    save_layer(
        centerline_gdf,
        "00_case_centerline"
    )


    centerline_length = float(
        centerline_geom.length
    )


except Exception as e:

    print(
        "Centerline generation failed:",
        e
    )

    centerline_length = None


print(
    "Centerline Length:",
    centerline_length
)

Saved layer: 00_case_centerline
Centerline Length: 2500.3900448895843


In [858]:
# ============================================================
# STEP 3
# CASE BUFFER
# ============================================================

case_buffer = case_gdf.copy()

BUFFER_AMOUNT = 330

case_buffer["geometry"] = (
    case_buffer.geometry.buffer(BUFFER_AMOUNT)
)

save_layer(
    case_buffer,
    "01_case_buffer"
)

Saved layer: 01_case_buffer


### Parks

In [859]:
def get_park_buffer(park_gdf):

    case_buffer = park_gdf.copy()

    geom = case_buffer.geometry.union_all()

    target_area = get_cooling_extent_area(case_buffer.area[0])

    original_area = geom.area

    def area_error(distance):
        buffered_geom = geom.buffer(distance)
        new_buffer_area = buffered_geom.area - original_area
        return new_buffer_area - target_area

    distance = brentq(
        area_error,
        0,
        10_000
    )

    result = geom.buffer(distance)

    return result

In [860]:
buffer_geometries = [get_park_buffer(park) for park in parks_gdfs]

In [861]:
buffer_geometries

[<POLYGON ((584304.852 4511632.957, 584522.135 4511505.966, 584527.579 451150...>,
 <POLYGON ((584119.373 4511160.463, 584124.106 4511157.698, 584128.553 451115...>]

In [862]:
parks_buffer = gpd.GeoDataFrame(
    geometry=buffer_geometries,
    crs=case_gdf.crs
)

In [863]:
parks_buffer

,geometry
0,"POLYGON ((584304.852 4511632.957, 584522.135 4..."
1,"POLYGON ((584119.373 4511160.463, 584124.106 4..."


In [864]:
# ============================================================
# STEP 4
# PARK BUFFER
# ============================================================

save_layer(
    parks_buffer,
    "02_park_buffer"
)

Saved layer: 02_park_buffer


### Water

In [865]:
water_gdfs

[[                                            geometry
  0  POLYGON ((584441.77 4512985.921, 584254.196 45...,
  'Large',
  'River']]

In [866]:
water_buffer_geometries = []

In [867]:
def get_water_buffer(water_gdf, water_size, water_type):

    case_buffer = water_gdf.copy()

    geom = case_buffer.geometry.union_all()

    distance = get_water_cooling_extent(water_size, water_type)

    result = geom.buffer(distance)

    return result

In [868]:
for gdf_list in water_gdfs:
    geometry = get_water_buffer(gdf_list[0], gdf_list[1], gdf_list[2])
    water_buffer_geometries.append(geometry)

In [869]:
water_gdf = gpd.GeoDataFrame(
    geometry=water_buffer_geometries,
    crs=case_gdf.crs
)

In [870]:
water_gdf

,geometry
0,"POLYGON ((585101.853 4513309.663, 585131.146 4..."


In [871]:
save_layer(
    water_gdf,
    "02_water_buffer"
)

Saved layer: 02_water_buffer


## Grid

In [872]:
# ============================================================
# STEP 5
# CREATE GRID
# ============================================================

xmin, ymin, xmax, ymax = (
    case_buffer.total_bounds
)

cell_size = 30

polys = []
row_indices = []
col_indices = []

row_idx = 0

for y in np.arange(
    ymin,
    ymax,
    cell_size
):

    col_idx = 0

    for x in np.arange(
        xmin,
        xmax,
        cell_size
    ):

        polys.append(
            box(
                x,
                y,
                x + cell_size,
                y + cell_size
            )
        )

        row_indices.append(
            row_idx
        )

        col_indices.append(
            col_idx
        )

        col_idx += 1

    row_idx += 1

grid = gpd.GeoDataFrame(
    {
        "fid": range(len(polys)),
        "row_index": row_indices,
        "col_index": col_indices
    },
    geometry=polys,
    crs=crs
)

save_layer(
    grid,
    "03_grid"
)

current_grid = grid.copy()

Saved layer: 03_grid


In [873]:
# ============================================================
# STEP 6
# INTERSECT CASE BUFFER
# ============================================================

case_buffer_union = union_geometry(
    case_buffer
)

current_grid = current_grid[
    current_grid.intersects(
        case_buffer_union
    )
].copy()

save_layer(
    current_grid,
    "04_grid_case_buffer"
)


Saved layer: 04_grid_case_buffer


In [874]:
# ============================================================
# STEP 7
# DISJOINT PARK BUFFER
# ============================================================

if parks_buffer is not None:

    park_union = union_geometry(
        parks_buffer
    )

    current_grid = current_grid[
        ~current_grid.intersects(
            park_union
        )
    ].copy()

    save_layer(
        current_grid,
        "05_grid_no_parks"
    )

Saved layer: 05_grid_no_parks


In [875]:
# ============================================================
# STEP 8
# DISJOINT WATER
# ============================================================

if water_gdf is not None:

    water_union = union_geometry(
        water_gdf
    )

    current_grid = current_grid[
        ~current_grid.intersects(
            water_union
        )
    ].copy()

    save_layer(
        current_grid,
        "06_grid_no_water"
    )
else:
    print("No water layer")

Saved layer: 06_grid_no_water


## Process points

In [876]:
# ============================================================
# STEP 9
# CENTROIDS
# ============================================================

centroids = current_grid.copy()

centroids["geometry"] = (
    centroids.centroid
)

save_layer(
    centroids,
    "07_centroids"
)

Saved layer: 07_centroids


In [877]:
# ============================================================
# STEP 10
# DISTANCE TO CASE STUDY POLYGON
# ============================================================

centroids["distance"] = (
    centroids.geometry.distance(
        case_polygon
    )
)

save_layer(
    centroids,
    "08_centroids_distance"
)

Saved layer: 08_centroids_distance


In [878]:
# ============================================================
# STEP 11
# REPROJECT TO WGS84
# ============================================================

centroids_wgs84 = (
    centroids.to_crs(
        epsg=4326
    )
)

save_layer(
    centroids_wgs84,
    "09_centroids_wgs84"
)

Saved layer: 09_centroids_wgs84


In [879]:
# ============================================================
# STEP 12
# ADD XY FIELDS
# ============================================================

centroids_wgs84["x"] = (
    centroids_wgs84.geometry.x
)

centroids_wgs84["y"] = (
    centroids_wgs84.geometry.y
)

centroids_wgs84 = centroids_wgs84.reset_index(drop=True)
centroids_wgs84 = centroids_wgs84.assign(fid=centroids_wgs84.index)

save_layer(
    centroids_wgs84,
    "10_centroids_xy"
)

Saved layer: 10_centroids_xy


In [880]:
np.isnan(xmin)

np.False_

## Extent

In [881]:
# ============================================================
# STEP 13
# CREATE EXTENT LAYER
# ============================================================

xmin, ymin, xmax, ymax = (
    current_grid.total_bounds
)

if (~np.isnan(xmin)):
    extent_poly = box(
        xmin,
        ymin,
        xmax,
        ymax
    )

    extent_gdf = gpd.GeoDataFrame(
        {
            "fid": [0]
        },
        geometry=[extent_poly],
        crs=crs
    )

    save_layer(
        extent_gdf,
        "11_extent"
    )

    # ============================================================
    # STEP 14
    # BUFFER EXTENT BY 50m
    # ============================================================

    extent_buffer = extent_gdf.copy()

    extent_buffer["geometry"] = (
        extent_buffer.geometry.buffer(50)
    )

    save_layer(
        extent_buffer,
        "12_extent_buffer"
    )

    # ============================================================
    # STEP 15
    # REPROJECT EXTENT TO WGS84
    # ============================================================

    extent_wgs84 = (
        extent_buffer.to_crs(
            epsg=4326
        )
    )

    save_layer(
        extent_wgs84,
        "13_extent_buffer_wgs84"
    )

    # ============================================================
    # STEP 16
    # REPORT EXTENT
    # ============================================================

    xmin, ymin, xmax, ymax = (
        extent_wgs84.total_bounds
    )

    print("\nFINAL EXTENT (WGS84)")
    print("--------------------")
    print("xmin:", xmin)
    print("ymin:", ymin)
    print("xmax:", xmax)
    print("ymax:", ymax)


Saved layer: 11_extent
Saved layer: 12_extent_buffer
Saved layer: 13_extent_buffer_wgs84

FINAL EXTENT (WGS84)
--------------------
xmin: -74.00025074243857
ymin: 40.749698297833625
xmax: -73.99724509494857
ymax: 40.753586829274305


## Metadata

In [882]:
# ============================================================
# STEP 17
# METADATA TABLE
# ============================================================

metadata_df = pd.DataFrame(
    {
        "key": [
            "total_area",
            "total_perimeter",
            "centerline_length",
            "candidate_count",
            "xmin",
            "ymin",
            "xmax",
            "ymax"
        ],
        "value": [
            total_area,
            total_perimeter,
            centerline_length,
            len(centroids_wgs84),
            xmin,
            ymin,
            xmax,
            ymax
        ]
    }
)

metadata_gdf = gpd.GeoDataFrame(
    metadata_df,
    geometry=[None] * len(metadata_df),
    crs=crs
)

save_layer(
    metadata_gdf,
    "metadata"
)

Saved layer: metadata


In [883]:
# ============================================================
# STEP 18
# METADATA JSON
# ============================================================

metadata = {

    "total_area":
        total_area,

    "total_perimeter":
        total_perimeter,

    "centerline_length":
        centerline_length,

    "candidate_count":
        int(
            len(
                centroids_wgs84
            )
        ),

    "extent_wgs84": {

        "xmin":
            float(xmin),

        "ymin":
            float(ymin),

        "xmax":
            float(xmax),

        "ymax":
            float(ymax)
    }
}

with open(
    metadata_json,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print("\nMetadata saved:")
print(metadata_json)

print("\nGeoPackage saved:")
print(output_gpkg)

print("\nWorkflow complete.")


Metadata saved:
/content/outputs/9/metadata.json

GeoPackage saved:
/content/outputs/9/workflow_outputs.gpkg

Workflow complete.


## Export Points

In [884]:
# ============================================================
# EXPORT POINTS
# ============================================================

columns = ['fid', 'row_index', 'col_index', 'distance', 'x', 'y']

centroids_wgs84[columns].to_csv(output_dir + '/points.csv', index=False)

# Download

In [885]:
import shutil

# Compress the folder into a zip file
# shutil.make_archive('output_zip_name', 'format', 'path_to_folder_to_zip')
shutil.make_archive('/content/outputs_zip', 'zip', '/content/outputs')

'/content/outputs_zip.zip'